<a href="https://colab.research.google.com/github/sensein/asd-ai-scoping-review/blob/update-scripts/scripts/PRISMA_pipeline_Fabio/local_filtering_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import pandas as pd
# Mount Google Drive
from google.colab import drive as gdrive
import re
import numpy as np
import statistics  # for calculating mean and standard deviation

In [ ]:
gdrive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the folder containing the subfolders with CSV files
main_folder = '/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries'

In [ ]:
#df_input_file = f'{main_folder}/autism_formatted__controlled_combined_noduplicates.csv'
df_input_file = f'{main_folder}/sciencedirect/autism_formatted.csv'

In [ ]:
df = pd.read_csv(df_input_file)
original_data = df
df

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year
0,Perspectives on children’s autistic traits in ...,Background\nWhile most autism research is cond...,"Autism spectrum disorder, Autism, Autistic tra...",https://doi.org/10.1016/j.ridd.2023.104576,https://www.sciencedirect.com/science/article/...,Elisa Genovesi and Philippa Ullmer and Laila B...,Research in Developmental Disabilities,2023
1,Chronic inhibition of astrocytic aquaporin-4 i...,"Social communication and interaction deficits,...","Autism spectrum disorder, Valproic acid, Aquap...",https://doi.org/10.1016/j.physbeh.2023.114286,https://www.sciencedirect.com/science/article/...,Shima Davoudi and Mona Rahdar and Narges Hosse...,Physiology & Behavior,2023
2,Touch in learning interactions with autistic c...,Learning is fundamentally based on the partici...,"Attention, Engagement, Instruction, Interperso...",https://doi.org/10.1016/j.lcsi.2023.100731,https://www.sciencedirect.com/science/article/...,Vivien Heller,"Learning, Culture and Social Interaction",2023
3,"Association of GABRG3, GABRB3, HTR2A gene vari...",Autism spectrum disorder (ASD) is a neurodevel...,"Autism (ASD), , , , RFLP",https://doi.org/10.1016/j.gene.2023.147399,https://www.sciencedirect.com/science/article/...,Ender M. Coskunpinar and Seymanur Tur and Nagi...,Gene,2023
4,Sialic acid and anti-ganglioside M1 antibodies...,Background\nAutism spectrum disorders (ASD) ar...,"Autism spectrum disorder, Autoimmune, Anti-GM1...",https://doi.org/10.1016/j.braindev.2022.11.006,https://www.sciencedirect.com/science/article/...,Engy A. Ashaat and Sahar Sabry and Moushira E....,Brain and Development,2023
...,...,...,...,...,...,...,...,...
8854,Perception et vécu de la maladie somatique sel...,Résumé\nLes nombreuses études sur la santé som...,"Insight, Maladie somatique, Prise en charge, P...",https://doi.org/10.1016/j.amp.2011.12.010,https://www.sciencedirect.com/science/article/...,Laura {Monduit de Caussade},"Annales Médico-psychologiques, revue psychiatr...",2013
8855,Regional brain volume abnormalities in Lesch-N...,Summary\nBackground\nLesch-Nyhan disease is a ...,NaN,https://doi.org/10.1016/S1474-4422(13)70238-2,https://www.sciencedirect.com/science/article/...,David J Schretlen and Mark Varvaris and Tiffan...,The Lancet Neurology,2013
8856,Le symptôme hypocondriaque dans un cas de psyc...,Résumé\nL’hypocondrie est un symptôme transver...,"Psychanalyse, Maternité, Psychose puerpérale, ...",https://doi.org/10.1016/j.evopsy.2012.08.013,https://www.sciencedirect.com/science/article/...,Marjorie Roques and Elisa Venturini and Gérard...,L'Évolution Psychiatrique,2013
8857,Identification of risk loci with shared effect...,Summary\nBackground\nFindings from family and ...,NaN,https://doi.org/10.1016/S0140-6736(12)62129-1,https://www.sciencedirect.com/science/article/...,NaN,The Lancet,2013


In [ ]:
"""
autism_words = ['autism', 'pervasive developmental disorder', 'autistic', 'asperger']
ai_words = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'machine-learning', 'deep-learning', 'ml', 'dl', 'cluster', 'clustered', 'clustering', 'classify', 'classified', 'classifying', 'classification', 'predict', 'predicted', 'predicting', 'prediction', 'recognize', 'recognized', 'recognizing', 'recognition', 'test', 'tested', 'testing', 'train', 'trained', 'training', 'model', 'modeled', 'modeling', 'supervised learning', 'unsupervised learning', 'reinforcement learning', 'self-supervised learning', 'transfer learning', 'zero-shot learning', 'few-shot learning', 'neural network', 'nn', 'transformer', 'convolutional network', 'convolutional neural network', 'cnn', 'computer vision', 'cv', 'natural language processing', 'nlp', 'audio signal processing', 'audio processing', 'asp', 'large language model', 'language model', 'llm', 'robot', 'robotics', 'agent', 'mining']
behavior_words = ['behavior', 'behave', 'behaved', 'behaving', 'behavioral', 'observation', 'observe', 'observing', 'observed',  'observational', 'pattern', 'response', 'respond', 'responded', 'responding', 'reaction', 'react', 'reacted', 'reacting', 'stereotype', 'stereotypical', 'stereotyped', 'repetitive', 'repeat', 'repeated', 'repeating', 'compulse', 'compulsive', 'tic', 'psychometrics', 'psychometric', 'phenotype', 'phenotypical', 'gaze', 'eye', 'fixation', 'motor', 'move', 'movement', 'moving', 'stand', 'standing', 'stood', 'crawl', 'crawled', 'crawling', 'walk', 'walking', 'walked', 'run', 'ran', 'jump', 'jumped', 'jumping', 'gait', 'locomotion', 'locomotory', 'body', 'posture', 'pose', 'gesture', 'face', 'facial', 'voice', 'speech', 'game', 'play', 'social', 'interaction', 'conversation', 'communication', 'emotion', 'emotional', 'stress', 'anxiety', 'rest', 'relax', 'sleep']
diagnosis_words = ['diagnosis', 'diagnostics', 'diagnostic', 'diagnose', 'diagnosing', 'diagnosed', 'screen', 'screened', 'screening', 'assessment', 'assess', 'assessing', 'assessed', 'detect', 'detection', 'detecting', 'detected']
"""

"\nautism_words = ['autism', 'pervasive developmental disorder', 'autistic', 'asperger']\nai_words = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'machine-learning', 'deep-learning', 'ml', 'dl', 'cluster', 'clustered', 'clustering', 'classify', 'classified', 'classifying', 'classification', 'predict', 'predicted', 'predicting', 'prediction', 'recognize', 'recognized', 'recognizing', 'recognition', 'test', 'tested', 'testing', 'train', 'trained', 'training', 'model', 'modeled', 'modeling', 'supervised learning', 'unsupervised learning', 'reinforcement learning', 'self-supervised learning', 'transfer learning', 'zero-shot learning', 'few-shot learning', 'neural network', 'nn', 'transformer', 'convolutional network', 'convolutional neural network', 'cnn', 'computer vision', 'cv', 'natural language processing', 'nlp', 'audio signal processing', 'audio processing', 'asp', 'large language model', 'language model', 'llm', 'robot', 'robotics', 'agent', 'mining']\nbeha

In [ ]:
autism_words = ['autism', 'pervasive developmental disorder', 'autistic', 'asperger']
autism_sigle = ['asc', 'asd', 'pdd', 'asds', 'ascs', 'pdds']
ai_words = ['ai', 'artificial intelligence', 'machine learning', 'deep learning', 'machine-learning', 'deep-learning', 'supervised learning', 'unsupervised learning', 'reinforcement learning', 'self-supervised learning', 'transfer learning', 'zero-shot learning', 'few-shot learning', 'neural network', 'transformer', 'convolutional network', 'convolutional neural network', 'cnn', 'computer vision', 'natural language processing', 'audio signal processing', 'audio processing', 'large language model', 'language model', 'robot', 'robotics']
behavior_words = ['behavior', 'behave', 'behaved', 'behaving', 'behavioral', 'observation', 'observe', 'observing', 'observed',  'observational', 'pattern', 'response', 'respond', 'responded', 'responding', 'reaction', 'react', 'reacted', 'reacting', 'stereotype', 'stereotypical', 'stereotyped', 'repetitive', 'repeat', 'repeated', 'repeating', 'compulse', 'compulsive', 'tic', 'psychometrics', 'psychometric', 'phenotype', 'phenotypical', 'gaze', 'eye', 'fixation', 'motor', 'move', 'movement', 'moving', 'stand', 'standing', 'stood', 'crawl', 'crawled', 'crawling', 'walk', 'walking', 'walked', 'run', 'ran', 'jump', 'jumped', 'jumping', 'gait', 'locomotion', 'locomotory', 'body', 'posture', 'pose', 'gesture', 'face', 'facial', 'voice', 'speech', 'game', 'play', 'social', 'interaction', 'conversation', 'communication', 'emotion', 'emotional', 'stress', 'anxiety', 'rest', 'relax', 'sleep']
diagnosis_words = ['diagnosis', 'diagnostics', 'diagnostic', 'diagnose', 'diagnosing', 'diagnosed', 'screen', 'screened', 'screening', 'assessment', 'assess', 'assessing', 'assessed', 'detect', 'detection', 'detecting', 'detected']

In [ ]:
#for filter_words in [autism_words, diagnosis_words, ai_words, behavior_words]:
for filter_words in [autism_words]:

  df = df[
      df['Title'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      df['Abstract'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      df['Keywords'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False)
  ]
  print(len(df))

7754


In [ ]:
removed_rows = original_data[~original_data.index.isin(df.index)]

# Check if there are removed rows
if not removed_rows.empty:
    # Randomly select one removed row and print it
    random_entry = removed_rows.sample()
    print(random_entry["Title"].values)
    print(random_entry["Abstract"].values)
    print(random_entry["Keywords"].values)
else:
    print("No entries were removed.")

['Reconstruction of X-rays spectra of clinical linear accelerators from transmission data with generalized simulated annealing']
['The spectral distribution of megavoltage X-rays used in radiotherapy departments is a fundamental quantity from which, in principle, all relevant information required for radiotherapy treatments can be determined. The direct measurement is difficult to achieve clinically and analyzing the transmission is a clinically viable indirect method for determining clinical linear accelerators photon spectra. In this method, transmission signals are acquired after the beam passes through different thicknesses of attenuators. The objective of this work was the establishment and application of an indirect method that used a spectral model based on generalized simulated annealing algorithm to determine the spectrum of clinical linear accelerators photons based on the transmission curve. Analysis of the spectra was made by analytical determination of dosimetric quantitie

In [ ]:
len(removed_rows)

1105

In [ ]:
for filter_words in [autism_sigle]:

  removed_rows = removed_rows[
      removed_rows['Title'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      removed_rows['Abstract'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False) |
      removed_rows['Keywords'].str.contains('|'.join(fr'\b{word}\b' for word in filter_words), case=False)
  ]
  print(len(removed_rows))

564


In [ ]:
removed_rows.to_csv(f'{main_folder}/sciencedirect/unrelated_acronyms_formatted.csv', index=False)

In [ ]:
removed_rows.shape

In [ ]:
# Check if there are removed rows
if not df.empty:
    # Randomly select one removed row and print it
    random_entry = df.sample()
    print(random_entry["Title"].values)
    print(random_entry["Abstract"].values)
    print(random_entry["Keywords"].values)
else:
    print("No entries were removed.")

In [ ]:
df = df.reset_index(drop=True)